<a href="https://colab.research.google.com/github/SwaksharDebnath/CRF-for-Bangla-Ancholik-NER-/blob/main/crf_for_mymensingh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install sbnltk
!pip install simpletransformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 690.7 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.8/330.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 58.2 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=75f1bc3913904be5a287b6abf8a2b6e66cca665d7ce9b9abbf40eb155a83bb54
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [ ]:
!pip install scikit-learn

In [ ]:
from sklearn.cluster import KMeans

In [ ]:
import numpy as np
import string

punctuations = list(string.punctuation) + ['।']
print(punctuations)

['!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', ':', ';', '<', '=', '>', '?', '@', '[', '\\', ']', '^', '_', '`', '{', '|', '}', '~', '।']


In [ ]:
def load_data(filename):
    with open(filename, 'r',encoding='utf-8') as f:
        data = [line.strip().split(' _ _ ') for line in f.readlines()]

        sentences = []
        cur = []
        for line in data:
            if line == ['']:
                cur = [tuple(line) for line in cur]
                sentences.append(cur)
                cur = []
            else:

                if len(line[0]) == 0:
                    line[0] = ' '
                cur.append(line)
        sentences.append(cur)
        return sentences

In [ ]:
def load_words(filename):
    with open(filename, 'r',encoding='utf-8') as f:
        data = [line.strip().split(' _ _ ') for line in f.readlines()]
        words = []
        for line in data:
            if line == ['']:
                continue
            else:
                words.append(line[0])
        # # convert each list to a tuple
        return list(set(words))

In [ ]:
train_words_set = load_words('/content/drive/MyDrive/NER_Dataset/ConvertedTxtData/train/Mymensingh_NER_train.txt')
test_words_set = load_words('/content/drive/MyDrive/NER_Dataset/ConvertedTxtData/test/Mymensingh_NER_test.txt')

In [ ]:
len(test_words_set), test_words_set[:10]

(1647,
 ['হাতও',
  'বত্\u200cশর',
  'হুনতে',
  'টুকরা',
  'নিজোরে',
  'চলো',
  'দেয়',
  'বাচ্চা',
  'লইছি',
  'টেবিলও'])

In [ ]:
!pip install fasttext

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 1.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pybind11-3.0.4-py3-none-any.whl.metadata (10 kB)
Using cached pybind11-3.0.4-py3-none-any.whl (314 kB)
  Created wheel for fasttext: filename=fasttext-0.9.3-cp312-cp312-linux_x86_64.whl size=4653915 sha256=d7fb3a21fe74ead4513f7f4f1169e59327964bacefd5fd6acac6ee9ef600680b
  Stored in directory: /root/.cache/pip/wheels/20/27/95/a7baf1b435f1cbde017cabdf1e9688526d2b0e929255a359c6
Successfully built fasttext


In [ ]:
from sbnltk.word_embedding import gensim_word2vec_embedding
w2v=gensim_word2vec_embedding()

Downloading...
From: https://drive.google.com/uc?id=142XvJg9xdpgzuYD31Y4pm-ZVdMaWmtuq
To: /usr/local/lib/python3.12/dist-packages/sbnltk/dataset/download_link.txt
100%|██████████| 1.66k/1.66k [00:00<00:00, 4.56MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1gF_b0a9BgroO8FpPUWWuSsCDeawYgEsb
From (redirected): https://drive.google.com/uc?id=1gF_b0a9BgroO8FpPUWWuSsCDeawYgEsb&confirm=t&uuid=dc6d2a1d-abd2-427b-a8f9-c51db28389ba
To: /usr/local/lib/python3.12/dist-packages/sbnltk/model/gensim_w2v.txt
100%|██████████| 1.89G/1.89G [00:19<00:00, 98.8MB/s]


In [ ]:
def get_w2v(words):
    ret = []
    for word in words:
        ret.append(w2v.get_vector(word))
    return np.asarray(ret)

def get_cluster_id(words):
    words_w2v = get_w2v(words)
    kmeans = KMeans(n_clusters=500, random_state=0).fit(words_w2v)
    return kmeans.labels_, kmeans

In [ ]:
train_words_w2v = get_w2v(train_words_set)

In [ ]:
test_words_w2v = get_w2v(test_words_set)

In [ ]:
kmeans = KMeans(n_clusters=500, random_state=0).fit(train_words_w2v)

In [ ]:
kmeans.labels_

array([ 0, 57], dtype=int32)

In [ ]:
kmeans.predict(train_words_w2v[0:2])

array([ 0, 57], dtype=int32)

In [ ]:
train = load_data('/content/drive/MyDrive/NER_Dataset/ConvertedTxtData/train/Mymensingh_NER_train.txt')

In [ ]:
test = load_data('/content/drive/MyDrive/NER_Dataset/ConvertedTxtData/test/Mymensingh_NER_test.txt')

In [ ]:
!pip install bnlp_toolkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 957.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.3/168.3 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 45.3 MB/s eta 0:00:00


In [ ]:
!git clone https://github.com/sagorbrur/bnlp

Cloning into 'bnlp'...
remote: Enumerating objects: 2353, done.
remote: Counting objects: 100% (160/160), done.
remote: Compressing objects: 100% (123/123), done.
remote: Total 2353 (delta 64), reused 93 (delta 34), pack-reused 2193 (from 3)
Receiving objects: 100% (2353/2353), 23.17 MiB | 19.77 MiB/s, done.
Resolving deltas: 100% (1334/1334), done.


In [ ]:
from bnlp import BengaliPOS

bn_pos = BengaliPOS(model_path="bnlp/model/bn_pos.pkl")

text = "আমি ভাত খাই।" # or you can pass ['আমি', 'ভাত', 'খাই', '।']
res = bn_pos.tag(text)
print(res)

[('আমি', 'PPR'), ('ভাত', 'NC'), ('খাই', 'VM'), ('।', 'PU')]


In [ ]:
train[0]

[('শুক্রবার', 'O'), ('আমি', 'O'), ('ওখানো', 'O'), ('যাইমু', 'O')]

In [ ]:
def add_pos(sentence):
    # first one of the tuple is the word and the second one is the ner
    words = []
    for word in sentence:
        words.append(word[0])
    words = " ".join(words)
    # print(words)
    pos = bn_pos.tag(words)
    ## add ner back to the tuple
    ret = []
    for i in range(len(pos)):
        word = sentence[i][0]
        ner = sentence[i][1]
        ret.append((word, pos[i][1], ner))
    return ret

def add_pos_to_all(sents):
    ret = []
    for i in range(len(sents)):
        if i % 100 == 0:
            print(i)
        ret.append(add_pos(sents[i]))
    return ret

In [ ]:
add_pos(train[0])

[('শুক্রবার', 'AMN', 'O'),
 ('আমি', 'PPR', 'O'),
 ('ওখানো', 'NV', 'O'),
 ('যাইমু', 'VM', 'O')]

In [ ]:
train_sents =  add_pos_to_all(train)

0
100
200
300
400
500
600
700
800
900
1000
1100
1200
1300
1400
1500
1600
1700
1800
1900
2000
2100
2200
2300
2400
2500
2600
2700


In [ ]:
test_sents = add_pos_to_all(test)

0
100
200
300
400
500
600


In [ ]:
train_sents[3]

[('তুমিতাইনরে', 'PPR', 'O'),
 ('ছাড়া', 'PP', 'O'),
 ('আমি', 'PPR', 'O'),
 ('বালা', 'NC', 'O'),
 ('নাই', 'CX', 'O')]

In [ ]:
test_sents[0]

[('মুই', 'NC', 'O'),
 ('আমার', 'PPR', 'O'),
 ('বন্ধুকে', 'NC', 'B-ROLE'),
 ('একটা', 'JQ', 'O'),
 ('লাল', 'JJ', 'B-COL'),
 ('গোলাফ', 'NC', 'B-OBJ'),
 ('কিনিয়া', 'CSB', 'O'),
 ('উপহার', 'NC', 'O'),
 ('দিয়াছি', 'VM', 'O')]

In [ ]:
def get_all_words(sents):
    words = []
    for i in range(len(sents)):
        for w in sents[i]:
            words += [w[0]]
    return words

In [ ]:
all_train_words = get_all_words(train_sents)
len(all_train_words)

16835

In [ ]:
word_freq = {}
for w in all_train_words:
    if w in word_freq:
        word_freq[w] += 1
    else:
        word_freq[w] = 1

In [ ]:
## feature extraction for Conditional Random Field - Bangla NER
def wordToFeatures(sent, idx):
    word = sent[idx][0]
    postag = sent[idx][1]

    cluster_id = kmeans.predict([w2v.get_vector(word)])[0]
    features = {
        'bias': 1.0,
        'word': word,
        'cluster_id': cluster_id,
        'word[-3:]': word[-3:],
        'word[-2:]': word[-2:],
        'word[:3]': word[:3],
        'word[:2]': word[:2],
        'word.isdigit': word.isdigit(),
        'index': idx,
        'length': len(word),
        'postag': postag,
        'freq': word_freq[word] if word in word_freq else 0,
    }


    for i in range(1, 3):
        if idx < i:
            break
        wordi = sent[idx-i][0]
        postagi = sent[idx-i][1]
        cluster_id_i = kmeans.predict([w2v.get_vector(wordi)])[0]
        features.update({
            '-{}:word'.format(i): wordi,
            '-{}:cluster_id'.format(i): cluster_id_i,
            '-{}:word[-3:]'.format(i): wordi[-3:],
            '-{}:word[-2:]'.format(i): wordi[-2:],
            '-{}:word[:3]'.format(i): wordi[:3],
            '-{}:word[:2]'.format(i): wordi[:2],
            '-{}:word.isdigit'.format(i): wordi.isdigit(),
            '-{}:postag'.format(i): postagi,
        })

    for i in range(1, 3):
        if (idx+i) >= len(sent):
            break
        wordi = sent[idx+i][0]
        postagi = sent[idx+i][1]
        cluster_id_i = kmeans.predict([w2v.get_vector(wordi)])[0]
        features.update({
            '{}:word'.format(i): wordi,
            '{}:cluster_id'.format(i): cluster_id_i,
            '{}:word[-3:]'.format(i): wordi[-3:],
            '{}:word[-2:]'.format(i): wordi[-2:],
            '{}:word[:3]'.format(i): wordi[:3],
            '{}:word[:2]'.format(i): wordi[:2],
            '{}:word.isdigit'.format(i): wordi.isdigit(),
            '{}:postag'.format(i): postagi,
        })

    if idx == 0:
        features['BOS'] = True
    if idx == len(sent) - 1:
        features['EOS'] = True

    return features

def sentTofeatures(sent):
    return [wordToFeatures(sent, i) for i in range(len(sent))]

def sentTolabels(sent):
    return [label for token, postag, label in sent]

In [ ]:
X_train = [sentTofeatures(s) for s in train_sents]
y_train = [sentTolabels(s) for s in train_sents]

In [ ]:
X_train[0][3]

{'bias': 1.0,
 'word': 'যাইমু',
 'cluster_id': np.int32(0),
 'word[-3:]': 'ইমু',
 'word[-2:]': 'মু',
 'word[:3]': 'যাই',
 'word[:2]': 'যা',
 'word.isdigit': False,
 'index': 3,
 'length': 5,
 'postag': 'VM',
 'freq': 18,
 '-1:word': 'ওখানো',
 '-1:cluster_id': np.int32(0),
 '-1:word[-3:]': 'ানো',
 '-1:word[-2:]': 'নো',
 '-1:word[:3]': 'ওখা',
 '-1:word[:2]': 'ওখ',
 '-1:word.isdigit': False,
 '-1:postag': 'NV',
 '-2:word': 'আমি',
 '-2:cluster_id': np.int32(89),
 '-2:word[-3:]': 'আমি',
 '-2:word[-2:]': 'মি',
 '-2:word[:3]': 'আমি',
 '-2:word[:2]': 'আম',
 '-2:word.isdigit': False,
 '-2:postag': 'PPR',
 'EOS': True}

In [ ]:
%%time
X_test = [sentTofeatures(s) for s in test_sents]
y_test = [sentTolabels(s) for s in test_sents]

CPU times: user 7.17 s, sys: 61.5 ms, total: 7.23 s
Wall time: 7.79 s


In [ ]:
import nltk
import sklearn
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelBinarizer
import sklearn_crfsuite as crfsuite
from sklearn_crfsuite import metrics

In [ ]:
crf = crfsuite.CRF(
    verbose='true',
    algorithm='lbfgs',
    c1=0.1,
    c2=0.1,
    max_iterations=150,
    all_possible_transitions=True
)
try:
    crf.fit(X_train, y_train, X_dev=X_test, y_dev=y_test)
except:
    pass

loading training data to CRFsuite: 100%|██████████| 2785/2785 [00:00<00:00, 3575.76it/s]


loading dev data to CRFsuite: 100%|██████████| 698/698 [00:00<00:00, 3410.28it/s]



Holdout group: 2

Feature generation
type: CRF1d
feature.minfreq: 0.000000
feature.possible_states: 0
feature.possible_transitions: 1
0....1....2....3....4....5....6....7....8....9....10
Number of features: 52237
Seconds required: 0.173

L-BFGS optimization
c1: 0.100000
c2: 0.100000
num_memories: 6
max_iterations: 150
epsilon: 0.000010
stop: 10
delta: 0.000010
linesearch: MoreThuente
linesearch.max_iterations: 20

Iter 1   time=0.37  loss=49502.62 active=51802 precision=0.047  recall=0.053  F1=0.050  Acc(item/seq)=0.895 0.615  feature_norm=0.12
Iter 2   time=0.09  loss=43789.74 active=49197 precision=0.047  recall=0.053  F1=0.050  Acc(item/seq)=0.895 0.615  feature_norm=0.11
Iter 3   time=0.37  loss=23198.22 active=37134 precision=0.051  recall=0.059  F1=0.051  Acc(item/seq)=0.759 0.263  feature_norm=0.05
Iter 4   time=0.15  loss=21081.17 active=45496 precision=0.047  recall=0.053  F1=0.050  Acc(item/seq)=0.895 0.615  feature_norm=0.06
Iter 5   time=0.08  loss=17918.02 active=46007 pr

In [ ]:
labels = list(crf.classes_)
labels

['O',
 'B-FOOD',
 'B-OBJ',
 'B-LOC',
 'B-PER',
 'B-REL',
 'I-REL',
 'I-FOOD',
 'B-ORG',
 'I-LOC',
 'B-COL',
 'I-COL',
 'B-ANI',
 'I-OBJ',
 'B-ROLE',
 'I-ROLE',
 'I-ANI',
 'I-ORG',
 'I-PER']

In [ ]:
y_pred = crf.predict(X_test)
metrics.flat_f1_score(y_test, y_pred,
                      average='macro', labels=labels)

0.5084696608617518